# 11 — Multi-Head Attention and the Transformer Block

**Description:** Build multi-head causal self-attention in PyTorch, inspect distinct heads, combine them with an output projection, and place attention inside a minimal Transformer block.
**Level:** Beginner
**Tags:** Language Models, Transformers, Multi-Head Attention, PyTorch, Residual Connections

A single attention head provides one learned routing pattern. A Transformer uses several heads so different representation subspaces can route information in parallel, then combines them with an output projection.

By the end, you will be able to:

- split the model width across attention heads;
- compute causal self-attention in parallel;
- concatenate heads and apply the output projection;
- compare single-head and multi-head outputs; and
- identify attention's place inside a minimal Transformer block.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.manual_seed(11)
plt.style.use("seaborn-v0_8-whitegrid")
torch.set_printoptions(precision=3, sci_mode=False)

## 1. Add batch and head dimensions

PyTorch modules normally receive `X` with shape `(batch, tokens, d_model)`. We choose $d_{model}=4$ and $H=2$ heads, so each head gets $d_{head}=d_{model}/H=2$ features.

In [ ]:
tokens = ["the", "robot", "fixed", "it"]
X = torch.tensor([[[
    1.0, 0.0, 0.2, 0.1
], [
    0.2, 1.0, 0.6, 0.0
], [
    0.1, 0.7, 1.0, 0.5
], [
    0.8, 0.2, 0.4, 1.0
]]])

B, T, d_model = X.shape
num_heads = 2
d_head = d_model // num_heads
print("X:", X.shape)
print("B, T, d_model, heads, d_head:", B, T, d_model, num_heads, d_head)
assert d_model % num_heads == 0

## 2. One projection creates all queries, keys, and values

Implementations commonly use one linear layer for efficiency. It produces $3d_{model}$ features, which we split into Q, K, and V. This is mathematically equivalent to three separate linear projections.

In [ ]:
qkv_projection = nn.Linear(d_model, 3 * d_model, bias=False)
qkv = qkv_projection(X)
Q, K, V = qkv.chunk(3, dim=-1)

print("combined QKV:", qkv.shape)
print("Q, K, V:   ", Q.shape, K.shape, V.shape)

## 3. Reshape features into heads

We reshape `(B, T, d_model)` into `(B, H, T, d_head)`. The transpose places the head axis before tokens, so matrix multiplication runs independently for every batch item and head.

In [ ]:
def split_heads(tensor, num_heads):
    B, T, width = tensor.shape
    d_head = width // num_heads
    return tensor.reshape(B, T, num_heads, d_head).transpose(1, 2)

Qh = split_heads(Q, num_heads)
Kh = split_heads(K, num_heads)
Vh = split_heads(V, num_heads)
print("per-head Q shape:", Qh.shape)  # (B, H, T, d_head)

## 4. Every head performs familiar attention

The last two dimensions contain the same calculation built in Notebooks 08–10. Broadcasting applies the causal mask to every batch item and head.

In [ ]:
scores = Qh @ Kh.transpose(-2, -1) / np.sqrt(d_head)
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(causal_mask, float("-inf"))
weights = torch.softmax(scores, dim=-1)
head_outputs = weights @ Vh

print("scores:      ", scores.shape)
print("weights:     ", weights.shape)
print("head outputs:", head_outputs.shape)
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, num_heads, T))

## 5. Visualize different heads

Each head has its own slice of the learned Q, K, and V projections, so its attention matrix can differ. Random initial weights do not yet encode useful language behavior; training gives the patterns purpose.

In [ ]:
fig, axes = plt.subplots(1, num_heads, figsize=(10, 4), constrained_layout=True)
for head, ax in enumerate(axes):
    matrix = weights[0, head].detach().numpy()
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(T), tokens)
    ax.set_yticks(range(T), tokens)
    ax.set(xlabel="key / source", ylabel="query / target", title=f"Head {head}")
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=axes, label="attention weight")
plt.show()

## 6. Concatenate heads

Each head produces `d_head` features. Move the token axis back and concatenate the head features to recover `d_model` features per token.

In [ ]:
def combine_heads(tensor):
    B, H, T, d_head = tensor.shape
    return tensor.transpose(1, 2).contiguous().reshape(B, T, H * d_head)

concatenated = combine_heads(head_outputs)
print("before combining:", head_outputs.shape)
print("after combining: ", concatenated.shape)

## 7. The output projection mixes head features

Concatenation only places features side by side. A learned matrix $W_O$ mixes information across heads and returns an update in model space.

In [ ]:
output_projection = nn.Linear(d_model, d_model, bias=False)
attention_output = output_projection(concatenated)

print("concatenated:   ", concatenated.shape)
print("W_O:            ", tuple(output_projection.weight.shape))
print("attention output:", attention_output.shape)

### Inspect the projection

Setting $W_O$ to the identity makes the output equal the concatenated heads. A learned non-diagonal matrix can recombine their features.

In [ ]:
identity_projection = nn.Linear(d_model, d_model, bias=False)
with torch.no_grad():
    identity_projection.weight.copy_(torch.eye(d_model))

assert torch.allclose(identity_projection(concatenated), concatenated)
print("identity output equals concatenation:", torch.allclose(identity_projection(concatenated), concatenated))

## 8. Package multi-head causal self-attention

**Self-attention** means Q, K, and V all come from the same input `x`. This module exposes the weights for inspection.

In [ ]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, return_weights=False):
        B, T, d_model = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        def to_heads(tensor):
            return tensor.reshape(B, T, self.num_heads, self.d_head).transpose(1, 2)

        q, k, v = map(to_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / (self.d_head ** 0.5)
        future = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        weights = torch.softmax(scores.masked_fill(future, float("-inf")), dim=-1)
        mixed = weights @ v
        mixed = mixed.transpose(1, 2).contiguous().reshape(B, T, d_model)
        output = self.out(mixed)
        return (output, weights) if return_weights else output

attention = MultiHeadCausalSelfAttention(d_model=4, num_heads=2)
output, learned_weights = attention(X, return_weights=True)
print("output:", output.shape)
print("weights:", learned_weights.shape)

## 9. Single head versus multiple heads

Keeping `d_model=4`, one head uses all four features in one routing pattern. Two heads use two features each and create two patterns. Both return the same external shape, so downstream layers can use either.

In [ ]:
torch.manual_seed(11)
single_head = MultiHeadCausalSelfAttention(d_model=4, num_heads=1)
torch.manual_seed(11)
two_heads = MultiHeadCausalSelfAttention(d_model=4, num_heads=2)

single_output, single_weights = single_head(X, return_weights=True)
multi_output, multi_weights = two_heads(X, return_weights=True)
print("single-head output:", single_output.shape, "weights:", single_weights.shape)
print("two-head output:   ", multi_output.shape, "weights:", multi_weights.shape)
print("outputs equal?     ", torch.allclose(single_output, multi_output))

Different head counts change how features are partitioned and how scores are scaled. Equal output shapes do not imply equal computations. More heads are not automatically better; head count is an architectural choice learned around during training.

## 10. Put attention inside a minimal Transformer block

Attention is one sublayer, not the entire block. A common **pre-norm** block has two residual updates:

$$x \leftarrow x + \operatorname{Attention}(\operatorname{LayerNorm}(x))$$

$$x \leftarrow x + \operatorname{MLP}(\operatorname{LayerNorm}(x))$$

Layer normalization stabilizes feature scales, residual paths preserve an information highway, and the MLP transforms each position independently.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_multiplier=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadCausalSelfAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_multiplier * d_model),
            nn.GELU(),
            nn.Linear(mlp_multiplier * d_model, d_model),
        )

    def forward(self, x):
        attention_update, weights = self.attention(self.norm1(x), return_weights=True)
        x = x + attention_update
        x = x + self.mlp(self.norm2(x))
        return x, weights

block = TransformerBlock(d_model=4, num_heads=2)
block_output, block_weights = block(X)
print("input shape: ", X.shape)
print("output shape:", block_output.shape)
assert block_output.shape == X.shape

The attention sublayer lets positions exchange information. The MLP works on each position separately. Residual connections keep the model width unchanged, allowing many blocks to be stacked.

## 11. Verify causality end to end

If we change the final input token, outputs at earlier positions should remain unchanged. The final output may change because it can attend to itself.

In [ ]:
X_changed = X.clone()
X_changed[:, -1] += 100.0

block.eval()
with torch.no_grad():
    original, _ = block(X)
    changed, _ = block(X_changed)

differences = (original - changed).abs().max(dim=-1).values.squeeze(0)
print("maximum difference at each position:", differences)
assert torch.allclose(original[:, :-1], changed[:, :-1], atol=1e-6)

## 12. Shape journey

| Stage | Shape |
| --- | --- |
| input | `(B, T, d_model)` |
| Q/K/V before splitting | `(B, T, d_model)` each |
| Q/K/V after splitting | `(B, H, T, d_head)` each |
| attention weights | `(B, H, T, T)` |
| per-head outputs | `(B, H, T, d_head)` |
| concatenated heads | `(B, T, d_model)` |
| output projection | `(B, T, d_model)` |

Writing these shapes beside an implementation is one of the fastest ways to diagnose attention bugs.

## 13. Challenges

1. Change `num_heads` to 4. What becomes of `d_head`?
2. Remove the output projection and explain what expressive operation is lost.
3. Add dropout to the attention weights and MLP for training.
4. Count the parameters in the attention module with `sum(p.numel() for p in attention.parameters())`.
5. Stack two blocks and verify that the shape remains unchanged.
6. Remove the causal mask and rerun the causality test. What fails?

## Series recap

The complete path is now visible:

```text
token IDs → embeddings → Q, K, V → scaled scores → causal mask
          → attention weights → weighted values → multiple heads
          → output projection → residual Transformer block → hidden states
          → logits → probabilities → sampled next token
```

Notebooks 01–06 explained the outer language-modeling loop. Notebooks 07–11 built the contextual computation at its core.

## Takeaways

- Multi-head attention runs several attention operations in parallel over feature subspaces.
- Each head can learn a different routing pattern.
- Concatenation restores model width; $W_O$ mixes features across heads.
- Causal self-attention prevents future tokens from influencing earlier positions.
- A Transformer block surrounds attention with normalization, residual connections, and a position-wise MLP.
- You now have every conceptual component of the core GPT-style Transformer block.